In [1]:
import sys
sys.path.append('../')

In [2]:
# from mdfc.compressor import MDFCompressor
# from mdfc.decompressor import MDFDecompressor
from mdfc import (MDFCompressor, MDFDecompressor)
import asammdf
import numpy as np  # assert allclose

In [3]:
# MDF_FIL = '../sample_data/nonrandom_sample_uncompressed.mf4'
# sample_data_path = '../sample_data/just_ints_uncompressed.mf4'
# MDFC_FIL = '../examples/nonrandom_sample_withzlib.mdfc'
# testfil = '../examples/just_ints.mdfc'


In [4]:
# from mdfc.report import generate_compression_report
# summary, details = generate_compression_report(testfil)
# summary.loc[['time_uncompressed_size', 'samples_uncompressed_size']].sum().sum()/8/1000/1000

In [5]:
from io import BytesIO
MDF_FIL = BytesIO()   # uncompressed MDF file
MDFC_FIL = BytesIO()  # compressed file buffer
from sample_data.generate_sample_data import generate_sample_file
generate_sample_file(
    MDF_FIL, 
    compression=False,
    include_sine_waves=True,
    include_keepalives=True,
    time_s=3600,
    sample_intervals_ms=(1000,500,250,100),
    channels_per_interval=50,
    channels_per_group=4
)
MDF_FIL.seek(0)
uncompressed_mdf_total_size = MDF_FIL.__sizeof__()

96 total groups are generated


In [6]:
# size of uncompressed MDF file in MB
f'{uncompressed_mdf_total_size/1000/1000:.3f} MB Uncompressed MDF File'

'48.020 MB Uncompressed MDF File'

In [7]:
# comparison against using deflate, 
#   which is in MDF standard
DEFLATE_MDF_FIL = BytesIO()
generate_sample_file(
    DEFLATE_MDF_FIL, 
    compression=1,
    include_sine_waves=True,
    include_keepalives=True,
    time_s=3600,
    sample_intervals_ms=(1000,500,250,100),
    channels_per_interval=50,
    channels_per_group=4
)
DEFLATE_MDF_FIL.seek(0)
deflate_mdf_total_size = DEFLATE_MDF_FIL.__sizeof__()

96 total groups are generated


In [8]:
# size of deflated MDF file in MB
f'{deflate_mdf_total_size/1000/1000:.3f} MB Deflate MDF File'

'30.148 MB Deflate MDF File'

In [9]:
# configurable parameters for lossy float compression
# for lossless fp compression, 
#   set tolerance & significands & minimum_tolerance
#   to -1, which are the function default values
# for testing, can set this to some falsey value
TEST_SIGNIFICANDS = 3  # None
TEST_MINIMUM_TOLERANCE = 1e-3  # None

# 
DO_TEST_COMPRESSION = True
DO_TEST_DECOMPRESSION = True

In [10]:
# %%timeit
# execute compression

# for lossless float compression,
#   significands, tolerance, and minimum_tolerance 
#   should not be passed
kws = (
    {}
    if not TEST_SIGNIFICANDS
    else (
        {'significands': TEST_SIGNIFICANDS}
    )
)
if TEST_SIGNIFICANDS and TEST_MINIMUM_TOLERANCE:
    kws['minimum_tolerance'] = TEST_MINIMUM_TOLERANCE

if DO_TEST_COMPRESSION:
    with (
        asammdf.MDF(MDF_FIL) as mfil,
        MDFCompressor(MDFC_FIL, close_file_on_exit=False) as cfil
    ):
        cfil.compress_all_signals(
            mfil,
            on_error='warn',
            **kws,
        )
        # presently, must call finish function,
        #   although perhaps that should be done in a successful __exit__?
        cfil.finish()
    MDFC_FIL.seek(0)

In [11]:
# size of compressed, MB
f'{MDFC_FIL.__sizeof__()/1000/1000:.3f} MB MDFC File'

'4.603 MB MDFC File'

In [12]:
# ratio of compressed vs uncompressed
overall_cr = (MDFC_FIL.__sizeof__() / MDF_FIL.__sizeof__())
print(f'Overall compression ratio is {overall_cr:.3f}, or {1/overall_cr:.2f}x')

Overall compression ratio is 0.096, or 10.43x


In [13]:
# ratio of compressed vs uncompressed
overall_cr = (MDFC_FIL.__sizeof__() / DEFLATE_MDF_FIL.__sizeof__())
print(f'Overall compression ratio vs Deflate is {overall_cr:.3f}, or {1/overall_cr:.2f}x')

Overall compression ratio vs Deflate is 0.153, or 6.55x


In [14]:
# %%timeit
# execute decompression & compare against original
if DO_TEST_DECOMPRESSION:
    samples_out = {}
    with (
        asammdf.MDF(MDF_FIL) as mfil,
        MDFDecompressor(MDFC_FIL, close_file_on_exit=False) as dfil
    ):
        try:
            # test signal decompression
            # for sn in dfil.metadata.keys():
            for sn in mfil.channels_db.keys():
                if sn == 'time': continue  #
                # compare against original
                original_sig = mfil.select([sn], raw=True)[0]
                original_timestamps = original_sig.timestamps
                original_samples = original_sig.samples
                
                # decompress mdfc signal
                res = dfil.decompress_signal(sn)
                samples_out[sn] = res

                # assert all close timestamps and values
                # timestamps... may have some minor losses
                #   due to float->scaleup->int on compression
                #   i think it should be understood that the retention
                #   is on the order of +/- 1 microsecond
                #       1e-7
                assert np.allclose(
                    original_timestamps,
                    samples_out[sn].timestamps,
                    atol=1e-7  # precise at microseconds
                ), f"{sn} timestamps not allclose!? :("
                assert np.allclose(
                    original_samples,
                    samples_out[sn].samples,
                    # tolerance specification for float case
                    # TODO perhaps this should be derived
                    #   from compression metadata,
                    #   ie the tolerance value used
                    atol=TEST_MINIMUM_TOLERANCE
                ), f"{sn} samples not allclose!? :("
        except KeyError:
            print(f'{sn} found in MDF but not in compressed file...')
            raise
        else:
            print("All signals have passed decompression check :)")
    MDFC_FIL.seek(0)


All signals have passed decompression check :)


In [15]:
# timecheck
# test_names = [
#     ... specific signal names...
# ]
test_names = None  # all

In [16]:
%%timeit
# testing the speed of reading MDF (without compression)
# data_path = '../sample_data/nonrandom_sample_uncompressed.mf4'
# data_path = '../sample_data/just_ints_uncompressed.mf4'
data_path = MDF_FIL
with (
    asammdf.MDF(data_path) as mfil,
):
    # test all signals except for time
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

116 ms ± 6.75 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [17]:
%%timeit
# testing the speed of reading MDF (with zlib compression)
# data_path = '../sample_data/nonrandom_sample_zlib.mf4'
# data_path = '../sample_data/just_ints_zlib.mf4'
data_path = DEFLATE_MDF_FIL
with (
    asammdf.MDF(data_path) as mfil,
):
    # test all signals except for time
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]
        

550 ms ± 37 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
%%timeit
# testing the speed of reading MDF (with deflate compression)
# data_path = '../sample_data/nonrandom_sample_deflate.mf4'
# data_path = '../sample_data/just_ints_deflate.mf4'
data_path = DEFLATE_MDF_FIL
with (
    asammdf.MDF(data_path) as mfil,
):
    # test all signals except for time
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]
        

515 ms ± 17.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit
# testing the speed of reading the MDFC compressed file
with MDFDecompressor(MDFC_FIL, close_file_on_exit=False) as dfil:
    if test_names is None:
        sigs = dfil.metadata.keys()
    else:
        sigs = test_names
    for sn in sigs:
        res = dfil.decompress_signal(sn)
    MDFC_FIL.seek(0)  # oops :)

154 ms ± 3.26 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
# file size benchmarks:
# 98200 kb size uncompressed         MDF file
# 56300 kb size compressed (deflate) MDF file
# 40700 kb size compressed (zlib)    MDF file
#  9100 kb size MDFC file with "super-duper compression"
#   which is (fastpfor, or zfp) + zlib_9
#   and 1e-3 fp minimum_tolerance 
#       highly likely this is reached in this example data
#       it's a sinewave from -1 to 1 with some jitter
#       therefore, fp compression is highly variable 
#       to this user input tolerance
# so, between 5-10x additional compression vs MDF standard!
#   of course, with lossy floating-point compression

In [ ]:
# read time benchmarks:
#  250 ms to read the uncompressed         MDF file
# 1100 ms to read the compressed (deflate) MDF file
# 1160 ms to read the compressed (zlib)    MDF file
#  335 ms to read the MDFC file with "super-duper compression"

In [ ]:
# so the advantages of this MDF compression utility are:
# *) better compression than the ASAM standard (MDF + deflate)
#       and option for lossy fp compression
#       specifying a tolerance (comprehensible setting by engineers)
# *) faster to read than compressed MDF file
#       and can be similar time as reading uncompressed file!
#       although perhaps this is due to MDF file structure?
#       something about sorting/unsorting/etc?
# *) immediate access to time metadata,
#       likely better memory management when "normalizing to dataframe"
#       although that wrapper isnt implemented yet
#       and may not be too important?
# *) columnar access each signal
#       compared to "group-access",
#       which may add more time to compress the bytes packet
#           which might contain multiple signals in the same CAN msg


# the disadvantages:
# *) requires special code/libraries
#       although these can/should be precompiled?
#       which will therefore also allow 
#           some level of cross-platform
# *) would require update to go with update to ASAM standard
#       to support new unique types/shapes
# *) may require a lot of memory allocation for comp/decomp
#       if we always do one-shot compression
#       something like... 2*<longest_signal>*64 bits required
#           maybe 3*... 
#       if thats... 10 hours at 100ms per
#           thats ~50 MB required
#           call it 250 MB?
#           not suitable for embedded, 
#           but could be OK as a server-side utility
# *) cannot be seamlessly (/natively) integrated with existing tools
#       eg ETAS-MDA software
#       ...this is not a disadvantage, IMO. 
#       this is a server-side utility, not an analysis tool